In [1]:
import numpy as np
import pandas as pd
import pickle
from _utils import find_wlres, noisy_arbitrary_funcs
from interrogator_hardware import InterrogatorHardware
from tqdm import tqdm

ModuleNotFoundError: No module named 'process_spectra'

In [2]:
# Import an optical source
optical_source = np.load('./data/source.npy')

# Create synthetic LPFGs for training

In [3]:
# wavelength for simulation
wl_sim = optical_source[:, 0]
# parameters for rand_function - limits
param = {'a':    [10, 40],            # depth
         'x0':   [1515e-9, 1585e-9],  # resonant wavelength
         'w':    [15e-9, 40e-9],      # fwhm
         'bias': [0, 10],             # insertion loss
         'fcn':  [0, 1]}

N = 25000 # number of spectra
k = 20    # number of noisy fbg per spectra
wl_bragg = []
X = []
X_ideal = []
wl_res = []
x0_size = int(N/100)
x0_strats = np.linspace(1515e-9, 1585e-9, x0_size)
dx0 = np.diff(x0_strats).mean()/2
base_position = np.linspace(1510e-9, 1590e-9, 13)
# d_bragg was set as 0.48 * np.mean(np.diff(base_position)) in synth_extended.dataset
# d_bragg was set asnp.mean(np.diff(base_position)) in synth_extended_dbragg_high.dataset
d_bragg = np.mean(np.diff(base_position))
fbg_shift = d_bragg #*0.48
for i in tqdm(range(N)):
    strat = x0_strats[i%x0_size]
    param['x0'] = [strat-dx0, strat+dx0]
    for j in range(k):
        params = []
        for key in param.keys():
            value = np.random.uniform(low=min(param[key]),
                                      high=max(param[key]))
            params.append(value)
            if key == 'x0':
                wl_res.append(value)
        spectra, ideal = noisy_arbitrary_funcs(wl_sim, *params)
        
        fbg_pos = base_position + np.random.uniform(low=-fbg_shift, high=fbg_shift, size=base_position.shape)
        interrogator_hardware = InterrogatorHardware(fbg_array=fbg_pos, 
                                                     fwhm=100e-12)
        wl_bragg.append(
            1e9 * interrogator_hardware.get_fbg_position()
        )
        
        x = interrogator_hardware.get_filtered_power(wl_sim, optical_source[:, 1] + spectra)
        x_id = interrogator_hardware.get_filtered_power(wl_sim, optical_source[:, 1] + ideal)
        x_f = interrogator_hardware.get_filtered_power(optical_source[:, 0], optical_source[:, 1])

        X.append(
            (x - x_f)/np.sum((x - x_f))
        )
        X_ideal.append(
            (x_id - x_f)/np.sum((x_id - x_f))
        )

100%|██████████| 25000/25000 [3:52:51<00:00,  1.79it/s]     


In [4]:
# Organize data
X = np.array(X)
X_ideal = np.array(X_ideal)
wl_bragg = np.array(wl_bragg)
wl_res = np.array(wl_res)
dataset = {'input_strength': X,
           'input_strength_clean': X_ideal,
           'wl_bragg': wl_bragg,
           'target': wl_res}

In [5]:
# Save data
with open('./data/synth_extended_dbragg_high.dataset', 'wb') as file:
    pickle.dump(dataset, file)

# Apply the interrogator to measured LPFGs

In [7]:
# Load measured spectra
with open('./data/measured_spectra.dataset', 'rb') as file:
    measured_spectra = pickle.load(file)

In [8]:
# Simulate the FBG array using the measured LPFG data
wl_bragg = []
X = []
wl_res = []
k = 100
base_position = np.linspace(1510e-9, 1590e-9, 13)
d_bragg = np.mean(np.diff(base_position))

for i in range(len(measured_spectra['transfer_fcn'])):
    for j in range(k):
        fbg_pos = base_position + np.random.uniform(low=-d_bragg*0.48, high=d_bragg*0.48, size=base_position.shape)
        interrogator_hardware = InterrogatorHardware(fbg_array=fbg_pos, 
                                                     fwhm=100e-12)
        wl_bragg.append(
            1e9 * interrogator_hardware.get_fbg_position()
        )
        wl_res.append(
            find_wlres(measured_spectra['wl']*1e-9, measured_spectra['transfer_fcn'][i])
        )
        
        x = interrogator_hardware.get_filtered_power(measured_spectra['wl']*1e-9, 
                                                     measured_spectra['transfer_fcn'][i]+np.interp(measured_spectra['wl']*1e-9, 
                                                                                                optical_source[:, 0], optical_source[:, 1]))
        x_f = interrogator_hardware.get_filtered_power(optical_source[:, 0], optical_source[:, 1])
        x_d = (x - x_f)
        X.append(
            x_d/np.sum(x_d)
        )

In [9]:
# Organize data
X = np.array(X)
wl_bragg = np.array(wl_bragg)
wl_res = np.array(wl_res)
dataset = {'input_strength': X,
           'wl_bragg': wl_bragg,
           'target': wl_res}

In [10]:
# Save data
with open('./data/measured.dataset', 'wb') as file:
    pickle.dump(dataset, file)